In [ ]:
import os
import gc
import time
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import cupy as cp
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, roc_auc_score, mean_squared_error, r2_score,
    precision_score, recall_score, f1_score, confusion_matrix, mean_absolute_error
)
from econml.dml import LinearDML
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression
# 修改：正确导入cuML的相关模块
from cuml.ensemble import RandomForestClassifier as cuRF, RandomForestRegressor as cuRFR
from cuml.linear_model import LogisticRegression as cuLogisticRegression, LinearRegression as cuLinearRegression
from cuml.svm import SVC as cuSVC, SVR as cuSVR
from cuml.model_selection import train_test_split as cu_train_test_split # 修正导入
from cuml.metrics import accuracy_score as cu_accuracy_score
from cuml import __version__ as cuml_version
import warnings
warnings.filterwarnings("ignore", message="Random state is currently ignored by probabilistic SVC")
# 新增：梯度提升模型导入
try:
    from cuml.ensemble import GradientBoostingClassifier as cuGBC, GradientBoostingRegressor as cuGBR
    HAS_GRADIENT_BOOSTING = True
except ImportError:
    print("cuML Gradient Boosting 不可用，将跳过该模型")
    HAS_GRADIENT_BOOSTING = False
# XGBoost导入
import xgboost as xgb
# 其他导入
from datetime import datetime
from pandas.api.types import CategoricalDtype
import traceback
from sklearn.base import clone, BaseEstimator
from sklearn.model_selection import KFold
from typing import Dict, Any, Tuple, Optional, List
import warnings
import json
warnings.filterwarnings('ignore')
# 新增：用于生成Word文档的库
from docx import Document
from docx.shared import Inches
# 新增：用于深度复制PyTorch模型
import copy
# 设置环境变量
os.environ['XGBOOST_DEVICE_MEMORY_LIMIT'] = '10GB'
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg') # 非交互模式，适合脚本
# ========================
# GPU资源清理增强模块
# ========================
def clear_gpu_resources():
    """全面清理GPU资源并添加延迟"""
    gc.collect()
    time.sleep(0.5)
   
    if 'cp' in globals():
        try:
            cp._default_memory_pool.free_all_blocks()
            cp.get_default_memory_pool().free_all_blocks()
        except Exception:
            pass
       
    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        except Exception:
            pass
def monitor_gpu_memory(threshold=0.8):
    """监控GPU内存使用，超过阈值则清理"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated()
        total = torch.cuda.get_device_properties(0).total_memory
        ratio = allocated / total
        if ratio > threshold:
            print(f"GPU内存使用率超过{threshold*100:.1f}% ({allocated/1e9:.2f}GB/{total/1e9:.2f}GB)，执行清理...")
            clear_gpu_resources()
            return True
    return False
# ========================
# 辅助函数：PyTorch版本的自助法标准误
# ========================
def bootstrap_se_torch(X: np.ndarray, y: np.ndarray, n_bootstrap: int = 1000) -> Tuple[float, float, float]:
    """
    使用自助法计算标准误和置信区间（PyTorch优化版）
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n_samples = len(y)
   
    bootstrap_effects = []
   
    for i in range(n_bootstrap):
        # 生成自助样本
        indices = np.random.choice(n_samples, n_samples, replace=True)
        X_boot = torch.tensor(X[indices], dtype=torch.float32, device=device)
        y_boot = torch.tensor(y[indices], dtype=torch.float32, device=device)
       
        # 使用正规方程求解
        with torch.no_grad():
            XtX = X_boot.t() @ X_boot
            Xty = X_boot.t() @ y_boot.reshape(-1, 1)
           
            # 添加小常数防止奇异矩阵
            XtX += torch.eye(X_boot.shape[1], device=device) * 1e-6
           
            coefficients = torch.linalg.solve(XtX, Xty)
            effect = coefficients[1].item() if coefficients.shape[0] > 1 else coefficients[0].item()
       
        bootstrap_effects.append(effect)
   
    # 计算标准误和置信区间
    bootstrap_effects = np.array(bootstrap_effects)
    se = np.std(bootstrap_effects, ddof=1)
    ci_lower = np.percentile(bootstrap_effects, 2.5)
    ci_upper = np.percentile(bootstrap_effects, 97.5)
   
    return se, ci_lower, ci_upper
# ========================
# 辅助函数：PyTorch版本的聚类稳健标准误
# ========================
def calculate_cluster_robust_se_torch(X: torch.Tensor,
                                     y: torch.Tensor,
                                     residuals: torch.Tensor,
                                     cluster_ids: np.ndarray) -> float:
    """
    计算聚类稳健标准误（PyTorch版本）
    """
    device = X.device
    unique_clusters = np.unique(cluster_ids)
    n_clusters = len(unique_clusters)
   
    scores = []
   
    for cluster in unique_clusters:
        cluster_mask = (cluster_ids == cluster)
        X_cluster = X[cluster_mask]
        residuals_cluster = residuals[cluster_mask]
       
        # 计算聚类得分
        cluster_score = X_cluster.t() @ residuals_cluster
        scores.append(cluster_score.cpu().numpy())
   
    # 计算聚类稳健方差估计
    scores_matrix = np.array(scores)
    mean_score = np.mean(scores_matrix, axis=0)
    cluster_variance = (scores_matrix - mean_score).T @ (scores_matrix - mean_score) / (n_clusters - 1)
   
    # 计算标准误
    XtX_inv = torch.linalg.inv(X.t() @ X).cpu().numpy()
    cluster_vcov = XtX_inv @ cluster_variance @ XtX_inv
    se = np.sqrt(np.diag(cluster_vcov))[1]
   
    return se
# ========================
# 新增：第一阶段模型工厂类,修改MLP实现
# ========================
class FirstStageModelFactory:
    """第一阶段模型工厂类，用于创建和配置不同的机器学习模型"""
    def __init__(self):
        self.model_configs = {
            'RandomForest': {
                'classification': lambda: cuRF(n_estimators=200, max_depth=20, random_state=42),
                'regression': lambda: cuRFR(n_estimators=200, max_depth=20, random_state=42)
            },
            'LogisticRegression': {
                'classification': lambda: cuLogisticRegression(penalty='l2', C=1.0, max_iter=1000), # 移除random_state
                'regression': lambda: cuLinearRegression(fit_intercept=True, algorithm='eig')
            },
            'GradientBoosting': {
                'classification': lambda: cuGBC(n_estimators=200, max_depth=6, random_state=42) if HAS_GRADIENT_BOOSTING else None,
                'regression': lambda: cuGBR(n_estimators=200, max_depth=6, random_state=42) if HAS_GRADIENT_BOOSTING else None
            },
            'SVC': {
                'classification': lambda: cuSVC(kernel='rbf', C=1.0, probability=True, random_state=42),
                'regression': lambda: cuSVR(kernel='rbf', C=1.0)
            },
            'XGBoost': {
                'classification': lambda: xgb.XGBClassifier(n_estimators=200, max_depth=6, random_state=42, tree_method='gpu_hist'),
                'regression': lambda: xgb.XGBRegressor(n_estimators=200, max_depth=6, random_state=42, tree_method='gpu_hist')
            },
            'MLP': {
                'classification': lambda input_size: PyTorchMLP(input_size=input_size, output_size=1, task_type='classification'),
                'regression': lambda input_size: PyTorchMLP(input_size=input_size, output_size=1, task_type='regression')
            }
        }
   
    def get_model(self, model_name, task_type, input_size=None):
        """
        根据模型名称和任务类型获取模型实例
        """
        if model_name not in self.model_configs:
            raise ValueError(f"不支持的模型类型: {model_name}")
       
        if task_type not in self.model_configs[model_name]:
            raise ValueError(f"模型 {model_name} 不支持任务类型: {task_type}")
       
        # 特殊处理MLP模型，需要输入维度
        if model_name == 'MLP':
            if input_size is None:
                raise ValueError("MLP模型需要指定input_size参数")
            return self.model_configs[model_name][task_type](input_size)
        else:
            model = self.model_configs[model_name][task_type]()
            if model is None:
                raise ValueError(f"模型 {model_name} 不可用，可能缺少依赖")
            return model
# ========================
# 新增：PyTorch MLP模型（GPU版本）修改 - 修正：减小batch_size，增加打印频率，添加内存监控
# ========================
class PyTorchMLP(nn.Module):
    """PyTorch多层感知机模型（GPU加速版），用于替代cuML的MLP"""
    def __init__(self, input_size, hidden_sizes=[100, 50], output_size=1,
                 task_type='classification', dropout_rate=0.1):
        super(PyTorchMLP, self).__init__()
        self.task_type = task_type
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
       
        # 构建网络层
        layers = []
        prev_size = input_size
       
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_size = hidden_size
       
        layers.append(nn.Linear(prev_size, output_size))
       
        # 根据任务类型添加输出层激活函数
        if task_type == 'classification':
            layers.append(nn.Sigmoid())
       
        self.network = nn.Sequential(*layers).to(self.device)
       
    def forward(self, x):
        return self.network(x)
   
    def fit(self, X, y, epochs=100, batch_size=128, lr=0.001):  # 修正：减小batch_size为128以减少内存使用
        """训练模型 - 修正：增加打印频率，每10epoch打印；每个epoch后监控内存"""
        self.train()
       
        # 转换数据为PyTorch张量
        if not isinstance(X, torch.Tensor):
            X_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
        else:
            X_tensor = X.to(self.device)
           
        if not isinstance(y, torch.Tensor):
            y_tensor = torch.tensor(y, dtype=torch.float32, device=self.device)
        else:
            y_tensor = y.to(self.device)
       
        # 确保输出维度匹配
        if self.task_type == 'classification':
            y_tensor = y_tensor.view(-1, 1)
       
        # 定义损失函数和优化器
        if self.task_type == 'classification':
            criterion = nn.BCELoss()
        else:
            criterion = nn.MSELoss()
       
        optimizer = optim.Adam(self.parameters(), lr=lr)
       
        # 训练循环
        dataset = torch.utils.data.TensorDataset(X_tensor, y_tensor)
        dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
       
        for epoch in range(epochs):
            epoch_loss = 0.0
            for batch_X, batch_y in dataloader:
                optimizer.zero_grad()
                outputs = self(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
           
            avg_loss = epoch_loss / len(dataloader)
            if (epoch + 1) % 10 == 0:  # 修正：每10epoch打印一次，增加频率
                print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")
           
            # 修正：每个epoch后监控并清理内存
            monitor_gpu_memory(threshold=0.85)  # 提高阈值到0.85，避免频繁清理
   
    def predict(self, X):
        """预测"""
        self.eval()
        with torch.no_grad():
            if not isinstance(X, torch.Tensor):
                X_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
            else:
                X_tensor = X.to(self.device)
               
            predictions = self(X_tensor)
           
            if self.task_type == 'classification':
                return (predictions > 0.5).float().cpu().numpy().flatten()
            else:
                return predictions.cpu().numpy().flatten()
   
    def predict_proba(self, X):
        """预测概率（仅分类任务）"""
        if self.task_type != 'classification':
            raise ValueError("predict_proba仅适用于分类任务")
       
        self.eval()
        with torch.no_grad():
            if not isinstance(X, torch.Tensor):
                X_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
            else:
                X_tensor = X.to(self.device)
               
            return self(X_tensor).cpu().numpy()
# ========================
# 新增：使用二折交叉验证计算残差的辅助函数
# ========================
def get_y_residuals(model_proto, X, y, n_splits=2):
    """
    使用二折交叉验证计算outcome (y) 的残差
    对于y，使用predict (标签) 计算残差
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    residuals = np.zeros(len(y))
   
    for train_idx, test_idx in kf.split(X):
        # 克隆模型
        if isinstance(model_proto, PyTorchMLP):
            model_clone = copy.deepcopy(model_proto)
        else:
            model_clone = clone(model_proto)
       
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
       
        # 训练克隆模型
        model_clone.fit(X_train, y_train)
       
        # 预测 (使用predict，得到标签)
        y_pred = model_clone.predict(X_test)
       
        # 计算残差
        residuals[test_idx] = y_test - y_pred
   
    return residuals
def get_t_residuals(model_proto, X, t, n_splits=2):
    """
    使用二折交叉验证计算treatment (t) 的残差
    对于t，如果有predict_proba，使用概率；否则使用predict
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    residuals = np.zeros(len(t))
   
    for train_idx, test_idx in kf.split(X):
        # 克隆模型
        if isinstance(model_proto, PyTorchMLP):
            model_clone = copy.deepcopy(model_proto)
        else:
            model_clone = clone(model_proto)
       
        X_train, X_test = X[train_idx], X[test_idx]
        t_train, t_test = t[train_idx], t[test_idx]
       
        # 训练克隆模型
        model_clone.fit(X_train, t_train)
       
        # 预测
        if hasattr(model_clone, 'predict_proba'):
            t_pred = model_clone.predict_proba(X_test)[:, 1] if model_clone.predict_proba(X_test).ndim > 1 else model_clone.predict_proba(X_test)
        else:
            t_pred = model_clone.predict(X_test)
       
        # 计算残差
        residuals[test_idx] = t_test - t_pred
   
    return residuals
# ========================
# 使用PyTorch进行DML分析（优化版）
# ========================
from scipy import stats
def second_stage_dml_analysis(t_residuals, y_residuals, X_second=None, robust_method='bootstrap', n_bootstrap=1000):
    """
    执行DML的第二阶段分析，计算处理效应及统计检验指标。
   
    参数:
        t_residuals (torch.Tensor或np.array): 第一阶段处理模型的残差。
        y_residuals (torch.Tensor或np.array): 第一阶段结果模型的残差。
        X_second (torch.Tensor或np.array, optional): 用于异质性分析的协变量。默认为None。
        robust_method (str): 稳健标准误方法，'bootstrap'或'hc1'。默认为'bootstrap'。
        n_bootstrap (int): Bootstrap重复次数，仅当robust_method='bootstrap'时有效。默认为1000。
   
    返回:
        dict: 包含效应值、标准误、置信区间、t统计量、p值、F统计量、F检验p值等的字典。
    """
    # 确保将Tensor转换为NumPy数组
    if torch.is_tensor(t_residuals):
        t_residuals = t_residuals.cpu().numpy() if t_residuals.is_cuda else t_residuals.numpy()
    if torch.is_tensor(y_residuals):
        y_residuals = y_residuals.cpu().numpy() if y_residuals.is_cuda else y_residuals.numpy()
    if X_second is not None and torch.is_tensor(X_second):
        X_second = X_second.cpu().numpy() if X_second.is_cuda else X_second.numpy()
   
    n = len(y_residuals)
   
    # 确保残差是NumPy数组并调整为正确的形状
    t_residuals = np.asarray(t_residuals).reshape(-1, 1)
    y_residuals = np.asarray(y_residuals).reshape(-1, 1)
   
    # 第二阶段回归：Y_residuals ~ theta * T_residuals + [其他交互项]
    if X_second is not None:
        # 如果有协变量用于异质性分析（CATE），包含交互项
        X_second = np.asarray(X_second)
        # 构建设计矩阵: [T_residuals, T_residuals * X_second]
        interaction_terms = t_residuals * X_second
        design_matrix = np.column_stack((t_residuals, interaction_terms))
    else:
        # 只估计ATE
        design_matrix = t_residuals
   
    # 使用OLS估计系数
    coefficients, _, _, _ = np.linalg.lstsq(design_matrix, y_residuals, rcond=None)
   
    # 提取处理效应（ATE是第一个系数）
    effect = coefficients[0][0] if len(coefficients) > 0 else 0.0
   
    # 计算预测值和残差
    y_pred = np.dot(design_matrix, coefficients)
    residuals = y_residuals - y_pred
   
    # 初始化变量
    se = 0.0
    t_statistic = 0.0
    p_value = 1.0
    f_statistic = 0.0
    f_p_value = 1.0
   
    # 计算标准误和置信区间
    if robust_method == 'bootstrap':
        bootstrap_effects = []
        for _ in range(n_bootstrap):
            indices = np.random.choice(n, n, replace=True)
            t_boot = t_residuals[indices]
            y_boot = y_residuals[indices]
            if X_second is not None:
                X_second_boot = X_second[indices]
                interaction_boot = t_boot * X_second_boot
                design_boot = np.column_stack((t_boot, interaction_boot))
            else:
                design_boot = t_boot
            coef_boot, _, _, _ = np.linalg.lstsq(design_boot, y_boot, rcond=None)
            bootstrap_effects.append(coef_boot[0][0] if len(coef_boot) > 0 else 0.0)
        se = np.std(bootstrap_effects, ddof=1) if bootstrap_effects else 0.0
    else:
        # 使用HC1稳健标准误的简化实现
        mse = np.sum(residuals**2) / (n - design_matrix.shape[1])
        xtx_inv = np.linalg.inv(np.dot(design_matrix.T, design_matrix))
        se_matrix = mse * xtx_inv
        se = np.sqrt(np.diag(se_matrix)[0]) if se_matrix.size > 0 else 0.0
   
    # 计算置信区间
    ci_lower = effect - 1.96 * se if se > 0 else effect
    ci_upper = effect + 1.96 * se if se > 0 else effect
   
    # 计算t统计量和p值 (只有在se>0时才有意义)
    if se > 0:
        t_statistic = effect / se
        df = n - design_matrix.shape[1] # 自由度
        p_value = 2 * (1 - stats.t.cdf(np.abs(t_statistic), df))
   
    # 计算F统计量和p值 (检验整个第二阶段模型是否显著)
    ss_total = np.sum((y_residuals - np.mean(y_residuals))**2)
    ss_residual = np.sum(residuals**2)
    if ss_residual > 0 and design_matrix.shape[1] > 0:
        ss_explained = ss_total - ss_residual
        df_model = design_matrix.shape[1]
        df_resid = n - df_model
        f_statistic = (ss_explained / df_model) / (ss_residual / df_resid) if df_resid > 0 else 0.0
        f_p_value = 1 - stats.f.cdf(f_statistic, df_model, df_resid) if df_resid > 0 else 1.0
   
    # 返回所有结果
    return {
        'effect': effect,
        'se': se,
        'lb': ci_lower,
        'ub': ci_upper,
        't_statistic': t_statistic,
        'p_value': p_value,
        'f_statistic': f_statistic,
        'f_p_value': f_p_value,
        'df': n - design_matrix.shape[1] if design_matrix.shape[1] > 0 else n
    }
def torch_dml_with_pretrained(X: np.ndarray,
                              y: np.ndarray,
                              t: np.ndarray,
                              model_y: BaseEstimator,
                              model_t: BaseEstimator,
                              use_robust_se: bool = True,
                              robust_method: str = 'bootstrap',
                              n_bootstrap: int = 1000,
                              cluster_ids: Optional[np.ndarray] = None,
                              n_splits: int = 2) -> Dict[str, Any]:
    """
    使用预训练的第一阶段模型执行DML的第二阶段分析（残差回归）
   
    参数:
        X: 协变量矩阵
        y: 结果变量
        t: 处理变量
        model_y: 预训练的结果变量模型
        model_t: 预训练的处理变量模型
        use_robust_se: 是否使用稳健标准误
        robust_method: 稳健标准误计算方法 ('bootstrap' 或 'hc1')
        n_bootstrap: 自助法重采样次数
        cluster_ids: 聚类ID（用于聚类稳健标准误）
        n_splits: 交叉验证折数
   
    返回:
        包含效应值、标准误、置信区间等结果的字典
    """
    print("执行DML第二阶段分析（残差回归）...")
   
    # 初始化所有变量
    effect = 0.0
    se = 0.0
    ci_lower = 0.0
    ci_upper = 0.0
    t_statistic = 0.0
    p_value = 1.0
    f_statistic = 0.0
    f_p_value = 1.0
    df = 0
   
    # 确保输入为NumPy数组
    X_np = X if isinstance(X, np.ndarray) else cp.asnumpy(X)
    y_np = y if isinstance(y, np.ndarray) else cp.asnumpy(y)
    t_np = t if isinstance(t, np.ndarray) else cp.asnumpy(t)
   
    # 第一阶段：使用指定折数交叉验证计算残差
    print(f"使用{n_splits}折交叉验证计算第一阶段残差...")
    y_residuals = get_y_residuals(model_y, X_np, y_np, n_splits=n_splits)
    t_residuals = get_t_residuals(model_t, X_np, t_np, n_splits=n_splits)
   
    # 转换为PyTorch张量
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    t_residuals_tensor = torch.tensor(t_residuals, dtype=torch.float32, device=device).reshape(-1, 1)
    y_residuals_tensor = torch.tensor(y_residuals, dtype=torch.float32, device=device)
   
    # 添加截距项
    ones = torch.ones_like(t_residuals_tensor, device=device)
    X_second = torch.cat([ones, t_residuals_tensor], dim=1)
   
    # 使用正规方程求解第二阶段系数
    with torch.no_grad():
        XtX = X_second.t() @ X_second
        Xty = X_second.t() @ y_residuals_tensor.reshape(-1, 1)
       
        # 添加小常数防止奇异矩阵
        XtX += torch.eye(2, device=device) * 1e-6
       
        coefficients = torch.linalg.solve(XtX, Xty)
        effect = coefficients[1].item() # 处理效应系数
   
    # 计算标准误和置信区间
    if use_robust_se and robust_method == 'bootstrap':
        print("使用自助法计算稳健标准误...")
        se, ci_lower, ci_upper = bootstrap_se_torch(
            X_second.cpu().numpy(),
            y_residuals_tensor.cpu().numpy(),
            n_bootstrap=n_bootstrap
        )
       
        # 计算t统计量和p值
        if se > 0:
            t_statistic = effect / se
            n, k = X_second.shape
            df = n - k
            p_value = 2 * (1 - stats.t.cdf(np.abs(t_statistic), df))
    else:
        # 计算经典标准误
        n, k = X_second.shape
        with torch.no_grad():
            y_pred_tensor = X_second @ coefficients
            residuals = y_residuals_tensor.reshape(-1, 1) - y_pred_tensor
        mse = torch.sum(residuals**2) / (n - k)
        cov_matrix = torch.linalg.inv(XtX) * mse
        se = torch.sqrt(torch.diag(cov_matrix))[1].item()
       
        # 计算置信区间
        t_stat_val = 1.96 # 95%置信水平的t统计量
        ci_lower = effect - t_stat_val * se
        ci_upper = effect + t_stat_val * se
       
        # 计算t统计量
        t_statistic = effect / se
        # 计算p值（双侧检验）
        df = n - k # 自由度
        p_value = 2 * (1 - stats.t.cdf(np.abs(t_statistic), df))
        # 计算F统计量
        with torch.no_grad():
            y_pred_tensor = X_second @ coefficients
            ss_residual = torch.sum((y_residuals_tensor.reshape(-1, 1) - y_pred_tensor)**2)
            ss_total = torch.sum((y_residuals_tensor.reshape(-1, 1) - torch.mean(y_residuals_tensor))**2)
            ss_explained = ss_total - ss_residual
   
            # F统计量 = (解释方差/自由度) / (残差方差/自由度)
            f_statistic = (ss_explained / (k - 1)) / (ss_residual / (n - k))
            f_p_value = 1 - stats.f.cdf(f_statistic, k - 1, n - k)
   
    print(f"DML处理效应估计: {effect:.6f}")
    print(f"标准误: {se:.6f}")
    print(f"95%置信区间: [{ci_lower:.6f}, {ci_upper:.6f}]")
    print(f"t统计量: {t_statistic:.4f}")
    print(f"p值: {p_value:.6f}")
    print(f"F统计量: {f_statistic:.4f}")
    print(f"F检验p值: {f_p_value:.6f}")
   
    return {
        'effect': effect,
        'se': se,
        'lb': ci_lower,
        'ub': ci_upper,
        't_statistic': t_statistic,
        'p_value': p_value,
        'f_statistic': f_statistic,
        'f_p_value': f_p_value,
        'df': df
    }
# ========================
# 修改：增强的第一阶段训练函数，尽量使用cuml - 修正：在训练后清理内存
# ========================
def first_stage_cuml(X, y, task_type='regression', model_name='RandomForest', test_size=0.25):
    """
    基础版第一阶段预测函数 - 使用新的评估函数
    """
    clear_gpu_resources()
   
    # 数据分割
    X_train, X_test, y_train, y_test = cu_train_test_split(
        X, y, test_size=test_size, random_state=42
    )
   
    # 创建模型
    model_factory = FirstStageModelFactory()
   
    try:
        model = model_factory.get_model(model_name, task_type)
       
        # 训练模型
        model.fit(X_train, y_train)
       
        # 预测
        y_pred = model.predict(X_test)
       
        # 使用新的评估函数计算指标 - 添加task_type参数
        metrics = evaluate_model(model, X_test, y_test, model_name, task_type)
        metrics['model_name'] = model_name
        metrics['task_type'] = task_type
       
        # 根据任务类型打印结果
        if task_type == 'classification':
            print(f"{model_name} {task_type}模型评估 - 准确率: {metrics.get('accuracy', 0):.4f}, F1: {metrics.get('f1', 0):.4f}, AUC: {metrics.get('auc', 0):.4f}")
        else:
            print(f"{model_name} {task_type}模型评估 - R²: {metrics.get('r2', 0):.4f}, RMSE: {metrics.get('rmse', 0):.4f}, MAE: {metrics.get('mae', 0):.4f}")
       
        # 修正：训练后清理内存
        clear_gpu_resources()
        monitor_gpu_memory(threshold=0.85)
       
        return model, metrics
       
    except Exception as e:
        print(f"训练{model_name}模型失败: {str(e)}")
        traceback.print_exc()
        return None, None
def first_stage_cuml_enhanced(X, y, task_type='regression', model_name='RandomForest', test_size=0.25):
    """
    增强版第一阶段预测，支持多种模型 - 使用新的评估函数
    """
    clear_gpu_resources()
   
    # 数据分割 - 使用cuML的train_test_split
    X_train, X_test, y_train, y_test = cu_train_test_split(
        X, y, test_size=test_size, random_state=42
    )
   
    # 创建模型
    model_factory = FirstStageModelFactory()
   
    try:
        # 特殊处理MLP模型，需要输入维度
        if model_name == 'MLP':
            input_size = X_train.shape[1]
            model = model_factory.get_model(model_name, task_type, input_size)
        else:
            model = model_factory.get_model(model_name, task_type)
       
        # 特殊处理PyTorch MLP模型
        if model_name == 'MLP':
            # 转换为PyTorch张量
            X_train_pt = torch.tensor(cp.asnumpy(X_train), dtype=torch.float32)
            y_train_pt = torch.tensor(cp.asnumpy(y_train), dtype=torch.float32)
           
            # 训练模型
            model.fit(X_train_pt, y_train_pt)
           
            # 预测
            X_test_pt = torch.tensor(cp.asnumpy(X_test), dtype=torch.float32)
            y_pred = model.predict(X_test_pt)
            y_pred = cp.array(y_pred)
        else:
            # 训练其他cuML/XGBoost模型
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
       
        # 使用新的评估函数计算评估指标 - 添加task_type参数
        metrics = evaluate_model(model, X_test, y_test, model_name, task_type)
       
        # 添加模型信息
        metrics['model_name'] = model_name
        metrics['task_type'] = task_type
       
        # 根据任务类型输出结果
        if task_type == 'classification':
            summary = f"准确率: {metrics.get('accuracy', 0):.4f}, F1: {metrics.get('f1', 0):.4f}, AUC: {metrics.get('auc', 0):.4f}"
        else:
            summary = f"R²: {metrics.get('r2', 0):.4f}, RMSE: {metrics.get('rmse', 0):.4f}, MAE: {metrics.get('mae', 0):.4f}"
       
        print(f"{model_name} {task_type}模型评估 - {summary}")
       
        # 修正：训练后清理内存
        clear_gpu_resources()
        monitor_gpu_memory(threshold=0.85)
       
        return model, metrics
       
    except Exception as e:
        print(f"训练{model_name}模型失败: {str(e)}")
        traceback.print_exc()
        return None, None
# ========================
# 新增：评估指标计算函数
# ========================
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
def evaluate_model(model, X_test, y_test, model_name="Model", task_type='classification'):
    """
    评估模型性能，根据任务类型使用不同的指标
   
    参数:
        model: 训练好的模型
        X_test: 测试特征
        y_test: 测试标签
        model_name: 模型名称
        task_type: 任务类型 ('regression' 或 'classification')
   
    返回:
        dict: 包含评估指标的字典
    """
    try:
        # 确保数据是NumPy数组（解决"Implicit conversion to a NumPy array is not allowed"错误）
        if hasattr(X_test, 'get'):
            X_test_np = X_test.get()
        elif hasattr(X_test, 'cpu'):
            X_test_np = X_test.cpu().numpy()
        elif hasattr(X_test, 'numpy'):
            X_test_np = X_test.numpy()
        else:
            X_test_np = np.array(X_test)
           
        if hasattr(y_test, 'get'):
            y_test_np = y_test.get()
        elif hasattr(y_test, 'cpu'):
            y_test_np = y_test.cpu().numpy()
        elif hasattr(y_test, 'numpy'):
            y_test_np = y_test.numpy()
        else:
            y_test_np = np.array(y_test)
       
        # 预测
        y_pred = model.predict(X_test_np)
       
        # 确保预测结果也是NumPy数组
        if hasattr(y_pred, 'get'):
            y_pred_np = y_pred.get()
        elif hasattr(y_pred, 'cpu'):
            y_pred_np = y_pred.cpu().numpy()
        elif hasattr(y_pred, 'numpy'):
            y_pred_np = y_pred.numpy()
        else:
            y_pred_np = np.array(y_pred)
       
        metrics = {}
       
        if task_type == 'regression':
            # 回归任务评估指标
            metrics['r2'] = r2_score(y_test_np, y_pred_np)
            metrics['mse'] = mean_squared_error(y_test_np, y_pred_np)
            metrics['rmse'] = np.sqrt(metrics['mse'])
            metrics['mae'] = mean_absolute_error(y_test_np, y_pred_np)
           
            print(f"{model_name} 回归模型评估 - R²: {metrics['r2']:.4f}, RMSE: {metrics['rmse']:.4f}, MAE: {metrics['mae']:.4f}")
           
        else:
            # 分类任务评估指标
            metrics['accuracy'] = accuracy_score(y_test_np, y_pred_np)
            metrics['precision'] = precision_score(y_test_np, y_pred_np, average='weighted')
            metrics['recall'] = recall_score(y_test_np, y_pred_np, average='weighted')
            metrics['f1'] = f1_score(y_test_np, y_pred_np, average='weighted')
           
            # 计算AUC
            auc_score = np.nan
            try:
                if hasattr(model, 'predict_proba'):
                    y_proba = model.predict_proba(X_test_np)
                    if hasattr(y_proba, 'get'):
                        y_proba = y_proba.get()
                    elif hasattr(y_proba, 'cpu'):
                        y_proba = y_proba.cpu().numpy()
                   
                    if len(y_proba.shape) > 1 and y_proba.shape[1] == 2: # 二分类
                        auc_score = roc_auc_score(y_test_np, y_proba[:, 1])
                    elif len(y_proba.shape) > 1: # 多分类
                        auc_score = roc_auc_score(y_test_np, y_proba, multi_class='ovr')
                    else: # 一维数组
                        auc_score = roc_auc_score(y_test_np, y_proba)
                elif hasattr(model, 'decision_function'):
                    y_score = model.decision_function(X_test_np)
                    if hasattr(y_score, 'get'):
                        y_score = y_score.get()
                    elif hasattr(y_score, 'cpu'):
                        y_score = y_score.cpu().numpy()
                    auc_score = roc_auc_score(y_test_np, y_score)
                else:
                    print(f"警告: 模型 {model_name} 不支持概率预测，无法计算AUC")
                    auc_score = np.nan
            except Exception as e:
                print(f"警告: 计算模型 {model_name} 的AUC时出错: {str(e)}")
                auc_score = np.nan
           
            metrics['auc'] = auc_score
            print(f"{model_name} 分类模型评估 - 准确率: {metrics['accuracy']:.4f}, 精确率: {metrics['precision']:.4f}, 召回率: {metrics['recall']:.4f}, F1: {metrics['f1']:.4f}, AUC: {metrics['auc']:.4f}")
       
        metrics['predictions'] = y_pred_np
        return metrics
       
    except Exception as e:
        print(f"评估模型 {model_name} 时出错: {str(e)}")
        traceback.print_exc()
        if task_type == 'regression':
            return {
                'r2': np.nan,
                'mse': np.nan,
                'rmse': np.nan,
                'mae': np.nan,
                'predictions': None
            }
        else:
            return {
                'accuracy': np.nan,
                'precision': np.nan,
                'recall': np.nan,
                'f1': np.nan,
                'auc': np.nan,
                'predictions': None
            }
# ========================
# 新增：模型比较器类
# ========================
class ModelComparator:
    """模型比较器，用于比较不同模型的性能并选择最佳模型"""
   
    def __init__(self, task_type):
        self.task_type = task_type
        self.model_metrics = {}
        self.weights = self._get_default_weights()
   
    def add_model(self, model_name, metrics):
        """添加模型及其评估指标"""
        self.model_metrics[model_name] = metrics
   
    def _get_default_weights(self):
        """获取默认指标权重"""
        if self.task_type == 'classification':
            return {'accuracy': 0.2, 'precision': 0.2, 'recall': 0.2, 'f1': 0.2, 'auc': 0.2}
        else:
            return {'r2': 0.4, 'rmse': 0.3, 'mae': 0.3}
   
    def _calculate_score(self, metrics):
        """计算模型综合得分（修复版）"""
        if self.task_type == 'regression':
            # 以R²为主要依据，同时考虑RMSE和MAE（数值越小越好，故取倒数）
            r2 = metrics.get('r2', 0)
            rmse = metrics.get('rmse', 1e6)
            mae = metrics.get('mae', 1e6)
            # 如果R²为负，给予惩罚性低分
            if r2 < 0:
                return -1.0 * abs(r2) # 或 return 0.0
            # 综合得分，R²权重最高
            score = r2 * 0.7 + (1 / (rmse + 1e-6)) * 0.15 + (1 / (mae + 1e-6)) * 0.15
            return score
        else:
            # 分类任务：可以考虑准确率、F1、AUC的加权平均
            accuracy = metrics.get('accuracy', 0)
            precision = metrics.get('precision', 0)
            recall = metrics.get('recall', 0)
            f1 = metrics.get('f1', 0)
            auc = metrics.get('auc', 0)
            if np.isnan(auc):
                auc = 0 # 将NaN的AUC视为0
            # 赋予AUC较高权重，如果AUC为0则主要依赖准确率和F1
            if auc > 0:
                score = accuracy * 0.2 + precision * 0.2 + recall * 0.2 + f1 * 0.2 + auc * 0.2
            else:
                score = accuracy * 0.25 + precision * 0.25 + recall * 0.25 + f1 * 0.25
            return score
   
    def compare_models(self):
        """比较所有模型并返回排名"""
        model_scores = {}
       
        for model_name, metrics in self.model_metrics.items():
            score = self._calculate_score(metrics)
            model_scores[model_name] = score
       
        # 按得分排序
        sorted_models = sorted(model_scores.items(), key=lambda x: x[1], reverse=True)
        return sorted_models
   
    def get_best_model(self):
        """获取最佳模型名称"""
        sorted_models = self.compare_models()
        return sorted_models[0][0] if sorted_models else None
   
    def print_comparison(self):
        """打印模型比较结果 - 使用新的评估函数"""
        print(f"\n=== {self.task_type.upper()} 模型比较结果 ===")
        sorted_models = self.compare_models()
       
        for rank, (model_name, score) in enumerate(sorted_models, 1):
            metrics = self.model_metrics[model_name]
           
            # 使用新的评估函数格式输出
            if self.task_type == 'classification':
                summary = f"准确率: {metrics.get('accuracy', 0):.4f}, 精确率: {metrics.get('precision', 0):.4f}, 召回率: {metrics.get('recall', 0):.4f}, F1: {metrics.get('f1', 0):.4f}, AUC: {metrics.get('auc', 0):.4f}"
            else:
                summary = f"R²: {metrics.get('r2', 0):.4f}, RMSE: {metrics.get('rmse', 0):.4f}, MAE: {metrics.get('mae', 0):.4f}"
           
            print(f"{rank}. {model_name}: 得分={score:.4f}, {summary}")
# ========================
# 敏感性分析类（优化版）修复函数引用）
# ========================
class SensitivityAnalysisOptimized:
    """敏感性分析类（优化版）"""
   
    def __init__(self, X: np.ndarray, y: np.ndarray, t: np.ndarray, control_variables: list):
        self.X = X
        self.y = y
        self.t = t
        self.control_variables = control_variables
        self.results = {}
    def run_analysis(self, variations: Dict[str, Dict[str, Any]], n_splits: int = 2) -> Dict[str, Any]:
        """运行敏感性分析（修复版）- 确保处理所有场景"""
        for name, variation in variations.items():
            print(f"运行敏感性分析: {name}")
           
            try:
                # 应用变异
                X_var = self.X
                if 'control_vars' in variation:
                    control_indices = [i for i, var in enumerate(self.control_variables)
                                     if var in variation['control_vars']]
                    X_var = self.X[:, control_indices]
               
                print("训练第一阶段模型...")
               
                # 使用增强版函数训练第一阶段模型
                model_y, metrics_y = first_stage_cuml_enhanced(
                    cp.array(X_var), cp.array(self.y),
                    task_type='classification',
                    model_name=variation.get('model_name', 'RandomForest')
                )
                model_t, metrics_t = first_stage_cuml_enhanced(
                    cp.array(X_var), cp.array(self.t),
                    task_type='classification',
                    model_name=variation.get('model_name', 'RandomForest')
                )
               
                # 计算残差（使用指定折数交叉验证）
                y_residuals = get_y_residuals(model_y, X_var, self.y, n_splits=n_splits)
                t_residuals = get_t_residuals(model_t, X_var, self.t, n_splits=n_splits)
               
                # 使用正确的第二阶段DML分析函数
                dml_result = second_stage_dml_analysis(
                    t_residuals=t_residuals,
                    y_residuals=y_residuals,
                    robust_method=variation.get('robust_method', 'bootstrap'),
                    n_bootstrap=variation.get('n_bootstrap', 1000)
                )
               
                # 提取所有结果，包括统计检验指标
                effect = dml_result['effect']
                se = dml_result['se']
                lb = dml_result['lb']
                ub = dml_result['ub']
                t_statistic = dml_result.get('t_statistic', np.nan) # 添加统计检验指标
                p_value = dml_result.get('p_value', np.nan)
                f_statistic = dml_result.get('f_statistic', np.nan)
                f_p_value = dml_result.get('f_p_value', np.nan)
               
                self.results[name] = {
                    'effect': effect,
                    'se': se,
                    'lb': lb,
                    'ub': ub,
                    't_statistic': t_statistic, # 添加统计检验指标
                    'p_value': p_value,
                    'f_statistic': f_statistic,
                    'f_p_value': f_p_value,
                    'metrics_y': metrics_y,
                    'metrics_t': metrics_t,
                    'variation': variation
                }
               
                print(f"{name}: 效应={effect:.6f}, 标准误={se:.6f}")
               
            except Exception as e:
                print(f"敏感性分析 {name} 出错: {str(e)}")
                traceback.print_exc()
                self.results[name] = {'error': str(e)}
       
        # 修正：将return语句移到循环外部，确保处理所有场景后才返回
        return self.results
   
    def summarize(self):
        """
        总结并格式化输出敏感性分析结果
        """
        summary_lines = []
        summary_lines.append("\n=== 敏感性分析结果总结 ===")
        summary_lines.append("场景\t\t处理效应\t标准误\t95%置信区间")
        summary_lines.append("-" * 65)
       
        for name, result in self.results.items():
            if 'error' not in result:
                effect = result['effect']
                se = result['se']
                lb = result['lb']
                ub = result['ub']
                summary_lines.append(f"{name[:12]:<12}\t{effect:.6f}\t{se:.6f}\t[{lb:.6f}, {ub:.6f}]")
            else:
                summary_lines.append(f"{name[:12]:<12}\t分析出错: {result['error']}")
       
        # 计算效应变化范围（如果所有分析都成功）
        successful_results = [res for res in self.results.values() if 'error' not in res]
        if len(successful_results) > 0:
            effects = [res['effect'] for res in successful_results]
            min_effect = min(effects)
            max_effect = max(effects)
            range_effect = max_effect - min_effect
            summary_lines.append("-" * 65)
            summary_lines.append(f"处理效应范围: {min_effect:.6f} 到 {max_effect:.6f} (范围: {range_effect:.6f})")
            summary_lines.append(f"效应变化幅度: {(range_effect / np.mean(effects)) * 100:.2f}%")
       
        return "\n".join(summary_lines)
# ========================
# 数据预处理封装
# ========================
class DataPreprocessor:
    """数据预处理类，封装数据读取和预处理逻辑"""
   
    def __init__(self, control_variables):
        self.control_variables = control_variables
        self.scalers = {}
        self.imputers = {}
        self.encoders = {}
   
    def load_and_preprocess_data(self, file_path, y_var, t_var):
        """加载并预处理数据"""
        print("步骤1: 数据准备")
        print(f"使用文件: {file_path}")
       
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"文件 '{file_path}' 不存在")
       
        # 读取数据
        data = pd.read_stata(file_path)
       
        # 处理因变量和处理变量
        y_series = data[y_var]
        t_series = data[t_var]
       
        # 编码和预处理
        le = LabelEncoder()
        y = le.fit_transform(y_series) if y_series.dtype == 'object' or isinstance(y_series.dtype, CategoricalDtype) else y_series.values
        t = le.fit_transform(t_series) if t_series.dtype == 'object' or isinstance(t_series.dtype, CategoricalDtype) else t_series.values
        # 检查所有控制变量是否存在于数据中
        missing_cols = [col for col in self.control_variables if col not in data.columns]
        if missing_cols:
            raise ValueError(f"数据中缺少以下控制变量: {missing_cols}")
       
        # 预处理控制变量
        for col in self.control_variables:
            # 处理分类变量
            if data[col].dtype == 'object' or isinstance(data[col].dtype, CategoricalDtype):
                self.encoders[col] = LabelEncoder()
                data[col] = self.encoders[col].fit_transform(data[col].astype(str))
           
            # 处理缺失值
            if data[col].isnull().sum() > 0:
                self.imputers[col] = SimpleImputer(strategy='mean')
                data[col] = self.imputers[col].fit_transform(data[col].values.reshape(-1, 1)).flatten()
           
            # 特征缩放
            self.scalers[col] = StandardScaler()
            data[col] = self.scalers[col].fit_transform(data[col].values.reshape(-1, 1)).flatten()
        # 创建特征矩阵
        X = data[self.control_variables].values
       
        # 转换为GPU数组
        X_gpu = cp.array(X, dtype=cp.float32)
        y_gpu = cp.array(y, dtype=cp.float32)
        t_gpu = cp.array(t, dtype=cp.float32)
       
        # 检查数据完整性
        print("检查数据完整性：")
        print("X 中是否有 NaN 或 Inf:", cp.any(cp.isnan(X_gpu)), cp.any(cp.isinf(X_gpu)))
        print("y 中是否有 NaN 或 Inf:", cp.any(cp.isnan(y_gpu)), cp.any(cp.isinf(y_gpu)))
        print("t 中是否有 NaN 或 Inf:", cp.any(cp.isnan(t_gpu)), cp.any(cp.isinf(t_gpu)))
       
        # 处理缺失值或无穷值
        X_gpu = cp.nan_to_num(X_gpu, nan=0.0, posinf=0.0, neginf=0.0)
        y_gpu = cp.nan_to_num(y_gpu, nan=0.0, posinf=0.0, neginf=0.0)
        t_gpu = cp.nan_to_num(t_gpu, nan=0.0, posinf=0.0, neginf=0.0)
       
        print(f"数据形状: X={X_gpu.shape}, y={y_gpu.shape}, t={t_gpu.shape}")
       
        return cp.asnumpy(X_gpu), cp.asnumpy(y_gpu), cp.asnumpy(t_gpu), data
# ========================
# 结果保存函数（增强版） - 修改为保存Word而不是CSV
# ========================
def save_detailed_results(effect, se, lb, ub, t_statistic, p_value, f_statistic, f_p_value, model_y, model_t, metrics_y, metrics_t,
                         sensitivity_results, save_path, robust_method='bootstrap',
                         batch_id="batch_001", prefix=""):
    """
    保存详细的结果到Word文件（修改版）- 确保敏感性分析结果正确合并
    """
    # 生成时间戳
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
   
    # 创建Word文档
    doc = Document()
    doc.add_heading(f'分析结果 - {batch_id}', 0)
   
    # 添加主要结果
    doc.add_heading('主要分析结果', level=1)
    para = doc.add_paragraph()
    para.add_run(f"处理效应估计: {effect:.6f}\n")
    para.add_run(f"标准误差: {se:.6f}\n")
    para.add_run(f"置信区间下界: {lb:.6f}\n")
    para.add_run(f"置信区间上界: {ub:.6f}\n")
    para.add_run(f"p值: {p_value:.6f}\n")
    para.add_run(f"t检验值: {t_statistic:.4f}\n")
    para.add_run(f"F检验值: {f_statistic:.4f}\n")
    para.add_run(f"f检验P值: {f_p_value:.6f}\n")
    para.add_run(f"Y模型类型: {type(model_y).__name__}\n")
    para.add_run(f"T模型类型: {type(model_t).__name__}\n")
    para.add_run(f"稳健方法: {robust_method}")
   
    # Y模型指标
    doc.add_heading('Y模型指标', level=2)
    para = doc.add_paragraph()
    para.add_run(f"准确率: {metrics_y.get('accuracy', np.nan):.4f}\n")
    para.add_run(f"精确率: {metrics_y.get('precision', np.nan):.4f}\n")
    para.add_run(f"召回率: {metrics_y.get('recall', np.nan):.4f}\n")
    para.add_run(f"F1: {metrics_y.get('f1', np.nan):.4f}\n")
    para.add_run(f"AUC: {metrics_y.get('auc', np.nan):.4f}")
   
    # T模型指标
    doc.add_heading('T模型指标', level=2)
    para = doc.add_paragraph()
    para.add_run(f"准确率: {metrics_t.get('accuracy', np.nan):.4f}\n")
    para.add_run(f"精确率: {metrics_t.get('precision', np.nan):.4f}\n")
    para.add_run(f"召回率: {metrics_t.get('recall', np.nan):.4f}\n")
    para.add_run(f"F1: {metrics_t.get('f1', np.nan):.4f}\n")
    para.add_run(f"AUC: {metrics_t.get('auc', np.nan):.4f}")
   
    # 敏感性分析结果
    doc.add_heading('敏感性分析结果', level=1)
    for name, result in sensitivity_results.items():
        if 'error' not in result:
            doc.add_heading(f'场景: {name}', level=2)
            para = doc.add_paragraph()
            para.add_run(f"处理效应: {result.get('effect', np.nan):.6f}\n")
            para.add_run(f"标准误差: {result.get('se', np.nan):.6f}\n")
            para.add_run(f"置信区间下界: {result.get('lb', np.nan):.6f}\n")
            para.add_run(f"置信区间上界: {result.get('ub', np.nan):.6f}\n")
            para.add_run(f"t统计量: {result.get('t_statistic', np.nan):.4f}\n")
            para.add_run(f"p值: {result.get('p_value', np.nan):.6f}\n")
            para.add_run(f"F统计量: {result.get('f_statistic', np.nan):.4f}\n")
            para.add_run(f"F检验p值: {result.get('f_p_value', np.nan):.6f}\n")
            para.add_run(f"稳健方法: {result.get('variation', {}).get('robust_method', robust_method)}")
           
            # Y模型指标
            metrics_y = result.get('metrics_y', {})
            para.add_run(f"\nY模型指标:\n")
            para.add_run(f"准确率: {metrics_y.get('accuracy', np.nan):.4f}\n")
            para.add_run(f"精确率: {metrics_y.get('precision', np.nan):.4f}\n")
            para.add_run(f"召回率: {metrics_y.get('recall', np.nan):.4f}\n")
            para.add_run(f"F1: {metrics_y.get('f1', np.nan):.4f}\n")
            para.add_run(f"AUC: {metrics_y.get('auc', np.nan):.4f}")
           
            # T模型指标
            metrics_t = result.get('metrics_t', {})
            para.add_run(f"\nT模型指标:\n")
            para.add_run(f"准确率: {metrics_t.get('accuracy', np.nan):.4f}\n")
            para.add_run(f"精确率: {metrics_t.get('precision', np.nan):.4f}\n")
            para.add_run(f"召回率: {metrics_t.get('recall', np.nan):.4f}\n")
            para.add_run(f"F1: {metrics_t.get('f1', np.nan):.4f}\n")
            para.add_run(f"AUC: {metrics_t.get('auc', np.nan):.4f}")
        else:
            para = doc.add_paragraph()
            para.add_run(f"场景 {name}: 错误 - {result['error']}")
   
    # 确保保存目录存在
    os.makedirs(save_path, exist_ok=True)
   
    # 保存Word文件
    word_file = os.path.join(save_path, f"{prefix}analysis_results_{batch_id}_{timestamp_str}.docx")
    doc.save(word_file)
    print(f"详细结果已保存到Word文件: {word_file}")
   
    return word_file
# ========================
# 新增：生成Word报告函数
# ========================
def generate_word_report(group_results, save_path, prefix=""):
    """生成Word报告，汇报所有计算结果"""
    doc = Document()
    doc.add_heading('DML分析报告', 0)
   
    for group_id, results in group_results.items():
        doc.add_heading(f'控制变量组 {group_id}', level=1)
       
        # 模型比较和选择
        doc.add_heading('Y模型比较和选择', level=2)
        for model_name, metrics in results['y_metrics'].items():
            if metrics is not None:
                para = doc.add_paragraph(f"模型 {model_name}: ")
                para.add_run(f"准确率: {metrics.get('accuracy', 'N/A'):.4f}, ")
                para.add_run(f"精确率: {metrics.get('precision', 'N/A'):.4f}, ")
                para.add_run(f"召回率: {metrics.get('recall', 'N/A'):.4f}, ")
                para.add_run(f"F1: {metrics.get('f1', 'N/A'):.4f}, ")
                para.add_run(f"AUC: {metrics.get('auc', 'N/A'):.4f}")
        doc.add_paragraph(f"选择的Y模型: {results['best_y_model']}")
       
        doc.add_heading('T模型比较和选择', level=2)
        for model_name, metrics in results['t_metrics'].items():
            if metrics is not None:
                para = doc.add_paragraph(f"模型 {model_name}: ")
                para.add_run(f"准确率: {metrics.get('accuracy', 'N/A'):.4f}, ")
                para.add_run(f"精确率: {metrics.get('precision', 'N/A'):.4f}, ")
                para.add_run(f"召回率: {metrics.get('recall', 'N/A'):.4f}, ")
                para.add_run(f"F1: {metrics.get('f1', 'N/A'):.4f}, ")
                para.add_run(f"AUC: {metrics.get('auc', 'N/A'):.4f}")
        doc.add_paragraph(f"选择的T模型: {results['best_t_model']}")
       
        # DML结果
        doc.add_heading('DML基准结果', level=2)
        dml_res = results['dml_results']
        para = doc.add_paragraph("处理效应估计: ")
        para.add_run(f"{dml_res.get('effect', 'N/A'):.6f}\n")
        para.add_run(f"标准误差: {dml_res.get('se', 'N/A'):.6f}\n")
        para.add_run(f"置信区间下界: {dml_res.get('lb', 'N/A'):.6f}\n")
        para.add_run(f"置信区间上界: {dml_res.get('ub', 'N/A'):.6f}\n")
        para.add_run(f"p值: {dml_res.get('p_value', 'N/A'):.6f}\n")
        para.add_run(f"t检验值: {dml_res.get('t_statistic', 'N/A'):.4f}\n")
        para.add_run(f"F检验值: {dml_res.get('f_statistic', 'N/A'):.4f}\n")
        para.add_run(f"f检验P值: {dml_res.get('f_p_value', 'N/A'):.6f}")
       
        # 敏感性分析
        doc.add_heading('敏感性分析结果', level=2)
        for sens_name, sens_res in results['sensitivity_results'].items():
            if 'error' not in sens_res:
                para = doc.add_paragraph(f"场景 {sens_name}: ")
                para.add_run(f"处理效应: {sens_res.get('effect', 'N/A'):.6f}, ")
                para.add_run(f"标准误差: {sens_res.get('se', 'N/A'):.6f}, ")
                para.add_run(f"置信区间下界: {sens_res.get('lb', 'N/A'):.6f}, ")
                para.add_run(f"置信区间上界: {sens_res.get('ub', 'N/A'):.6f}, ")
                para.add_run(f"t统计量: {sens_res.get('t_statistic', 'N/A'):.4f}, ")
                para.add_run(f"p值: {sens_res.get('p_value', 'N/A'):.6f}, ")
                para.add_run(f"F统计量: {sens_res.get('f_statistic', 'N/A'):.4f}, ")
                para.add_run(f"F检验p值: {sens_res.get('f_p_value', 'N/A'):.6f}")
               
                # Y模型指标
                metrics_y = sens_res.get('metrics_y', {})
                para.add_run(f"\nY模型指标 - 准确率: {metrics_y.get('accuracy', 'N/A'):.4f}, 精确率: {metrics_y.get('precision', 'N/A'):.4f}, 召回率: {metrics_y.get('recall', 'N/A'):.4f}, F1: {metrics_y.get('f1', 'N/A'):.4f}, AUC: {metrics_y.get('auc', 'N/A'):.4f}")
               
                # T模型指标
                metrics_t = sens_res.get('metrics_t', {})
                para.add_run(f"\nT模型指标 - 准确率: {metrics_t.get('accuracy', 'N/A'):.4f}, 精确率: {metrics_t.get('precision', 'N/A'):.4f}, 召回率: {metrics_t.get('recall', 'N/A'):.4f}, F1: {metrics_t.get('f1', 'N/A'):.4f}, AUC: {metrics_t.get('auc', 'N/A'):.4f}")
            else:
                doc.add_paragraph(f"场景 {sens_name}: 错误 - {sens_res['error']}")
       
        # 敏感性总结
        doc.add_paragraph(results['sensitivity_summary'])
   
    # 保存Word文档
    word_file = os.path.join(save_path, f"{prefix}dml_analysis_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.docx")
    doc.save(word_file)
    print(f"Word报告已生成: {word_file}")
    return word_file
# ========================
# 新增：极坐标柱状图函数 - 修改版
# ========================
from matplotlib.lines import Line2D
from matplotlib import cm
def plot_polar_bars(data, num_metrics, num_models, save_path, filename):
    """
    绘制极坐标柱状图，展示各模型在不同指标上的表现，并在顶部标注模型名称。
   
    参数:
    - data: 包含模型得分的 DataFrame，必须包含 'Model' 列和指标列
    - num_metrics: 指标数量
    - num_models: 模型数量
    - save_path: 保存路径
    - filename: 保存文件名
    """
    models = data['Model'].tolist()
    metrics = data.columns[:-1].tolist() # 指标列
    # 检查输入的指标和模型数量是否正确
    assert len(metrics) == num_metrics, f"Expected {num_metrics} metrics, but got {len(metrics)}"
    assert len(models) == num_models, f"Expected {num_models} models, but got {len(models)}"
    # 提取原始数据
    values_array = data.iloc[:, :-1].values
    # 处理NaN值：将NaN替换为0，以避免绘图错误
    values_array = np.nan_to_num(values_array, nan=0.0)
    min_visible_height = 0.6 
    max_height = 1.6 
    for j in range(values_array.shape[1]): # 对于每个指标（列）
        vals = values_array[:, j]
        if np.all(vals == 0): # 如果所有值为0，跳过
            continue
        min_v = np.min(vals)
        max_v = np.max(vals)
        if max_v > min_v:
            normalized = (vals - min_v) / (max_v - min_v)
        else:
            normalized = np.full_like(vals, 0.5)
            for i in range(len(normalized)):
                offset = (i - (len(normalized) - 1) / 2) * 0.1 / (len(normalized) - 1) # 线性分布小偏移
                normalized[i] += offset
            # 归一化回[0,1]
            normalized = (normalized - np.min(normalized)) / (np.max(normalized) - np.min(normalized) + 1e-6)
        # 映射到[min_visible_height, max_height]
        scaled = min_visible_height + normalized * (max_height - min_visible_height)
        values_array[:, j] = scaled
    for i in range(values_array.shape[0]): # 对于每个模型（行）
        vals = values_array[i, :]
        # 应用简单移动平均平滑（窗口大小2，边界复制）
        if len(vals) >= 2:
            smoothed = np.convolve(vals, np.ones(2)/2, mode='same')
            # 边界处理：第一个和最后一个保持原值以保留差异
            smoothed[0] = vals[0]
            smoothed[-1] = vals[-1]
            values_array[i, :] = smoothed
    # 更新DataFrame以使用调整后的值
    data.iloc[:, :-1] = values_array
    # 设置极坐标图
    angles = np.linspace(0, 2 * np.pi, num_models, endpoint=False).tolist()
    angles += angles[:1] # 闭合图形
    # 设置颜色
    colors = cm.viridis(np.linspace(0, 1, num_models))
    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
    outer_radius = max_height + 0.2 
    inner_radius = 0.2
    # 每个柱的宽度 - 无间隙，柱子连接起来
    bar_width = (2 * np.pi / num_models) / num_metrics
    # 绘制柱状图
    for i, model in enumerate(models):
        values = data.loc[data['Model'] == model, metrics].values.flatten().tolist()
        angle = angles[i]
        for j, value in enumerate(values):
            offset_angle = angle + j * bar_width
            ax.bar(offset_angle, value, bottom=inner_radius, width=bar_width, label=models[i] if j == 0 else "",
                   alpha=0.7, color=colors[i])
            # 调整标签位置和旋转 - 放在柱子顶部，径向旋转以避免重叠
            text_radius = inner_radius + value + 0.05
            text_rotation = np.degrees(offset_angle) - 90 # 径向向外旋转
            ax.text(offset_angle, text_radius, metrics[j],
                    ha='center', va='bottom', fontsize=8, rotation=text_rotation, rotation_mode='anchor')
    # 设置极坐标轴属性
    ax.set_ylim(0, outer_radius)
    ax.set_yticklabels([])
    ax.set_xticks([])
    # 添加图例（移到右上角）
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor=colors[i], markersize=10, label=models[i])
        for i in range(num_models)
    ]
    ax.legend(handles=legend_elements, title='Models', title_fontsize=12,
              bbox_to_anchor=(1.1, 1.0), loc='upper right', fontsize=10)
    ax.spines['polar'].set_visible(False)
    plt.tight_layout()
    full_path = os.path.join(save_path, f"{filename}.pdf")
    plt.savefig(full_path, format='pdf', bbox_inches='tight', dpi=1200)
    plt.close()
    print(f"极坐标图已保存到: {full_path}")
# ========================
# 修改：主程序逻辑（增强版） - 支持多组控制变量并生成Word报告
# ========================
if __name__ == "__main__":
    try:
        # 初始化时限制GPU内存
        if torch.cuda.is_available():
            torch.cuda.set_per_process_memory_fraction(0.8)
           
        print("正在执行DML处理流程（多模型比较版）")
        print(f"cuML版本: {cuml_version}")
       
        # 定义大组
        groups = {
            '基准回归': {'t_var': 'mobile_use', 'y_var': 'entrepreneurship', 'prefix': '基准回归_'},
            '替换自变量': {'t_var': 'computer_use', 'y_var': 'entrepreneurship', 'prefix': '替换自变量_'},
            '替换因变量': {'t_var': 'mobile_use', 'y_var': 'fam_enter', 'prefix': '替换因变量_'}
        }
       
        # 定义控制变量组（相同于所有大组）
        control_groups = {
          1: [
                'contracts', 'ln_total_asset', 'impro_sasti', 'Child',
                'lifefire', 'age', 'ln_renqing', 'cfpsedu', 'laborasso', 'familysize',
                'Oldage', 'health', 'marriage', 'army', 'ln_finc', 'exercise',
                'intelligence', 'person_status', 'party', 'workasso',
                'fid','pid','year'
            ],
            2: [
                'contracts', 'ln_total_asset', 'impro_sasti', 'Child',
                'lifefire', 'age', 'ln_renqing', 'cfpsedu', 'laborasso', 'familysize',
                'Oldage', 'health', 'marriage', 'army', 'ln_finc', 'exercise',
                'intelligence', 'person_status', 'party', 'workasso',
                'fid','pid','year',
                'ln_total_asset_sq', 'impro_sasti_sq', 'Child_sq',
                'lifefire_sq', 'age_sq', 'ln_renqing_sq', 'cfpsedu_sq', 'familysize_sq',
                'Oldage_sq', 'health_sq', 'marriage_sq', 'ln_finc_sq', 'exercise_sq',
                'intelligence_sq', 'person_status_sq'
            ],
            3: [
                'contracts', 'ln_total_asset', 'impro_sasti', 'Child',
                'lifefire', 'age', 'ln_renqing', 'cfpsedu', 'laborasso', 'familysize',
                'Oldage', 'health', 'marriage', 'army', 'ln_finc', 'exercise',
                'intelligence', 'person_status', 'party', 'workasso',
                'fid','pid','year',
                'ln_total_asset_sq', 'impro_sasti_sq', 'Child_sq',
                'lifefire_sq', 'age_sq', 'ln_renqing_sq', 'cfpsedu_sq', 'familysize_sq',
                'Oldage_sq', 'health_sq', 'marriage_sq', 'ln_finc_sq', 'exercise_sq',
                'intelligence_sq', 'person_status_sq',
                'ln_total_asset_cub', 'impro_sasti_cub', 'Child_cub',
                'lifefire_cub', 'age_cub', 'ln_renqing_cub', 'cfpsedu_cub', 'familysize_cub',
                'Oldage_cub', 'health_cub', 'marriage_cub', 'ln_finc_cub', 'exercise_cub',
                'intelligence_cub', 'person_status_cub'
            ]
        }
       
        # 定义折数
        folds = [2,5]
       
        # 定义要比较的模型列表
        model_names = ['RandomForest', 'LogisticRegression', 'SVC', 'XGBoost', 'MLP']
        if HAS_GRADIENT_BOOSTING:
            model_names.append('GradientBoosting')
       
        # 保存路径
        save_path = "D:\\my-rapids-project\\result"
        os.makedirs(save_path, exist_ok=True)
       
        # 收集所有组的结果，用于Word报告
        all_group_results = {}
       
        for group_name, group_config in groups.items():
            print(f"\n=== 处理大组: {group_name} ===")
            group_results = {}
            group_counter = 1 # 用于唯一标识每个组合
           
            for fold in folds:
                print(f"\n=== 处理折数: {fold} ===")
                for group_id, control_names in control_groups.items():
                    combo_id = f"fold_{fold}_group_{group_id}"
                    print(f"\n=== 处理组合 {combo_id} ===")
                   
                    # 数据预处理
                    preprocessor = DataPreprocessor(control_names)
                    X, y, t, original_data = preprocessor.load_and_preprocess_data("l3dml_data_digital.dta", group_config['y_var'], group_config['t_var'])
                   
                    # 转换为GPU数组
                    X_gpu = cp.array(X, dtype=cp.float32)
                    y_gpu = cp.array(y, dtype=cp.float32)
                    t_gpu = cp.array(t, dtype=cp.float32)
                   
                    # 创建模型比较器
                    y_comparator = ModelComparator('classification')
                    t_comparator = ModelComparator('classification')
                   
                    # 训练并比较Y模型（分类任务）
                    print("\n=== 训练并比较Y模型（分类任务）===")
                    y_models = {}
                    y_metrics = {}
                    for model_name in model_names:
                        try:
                            print(f"\n训练Y模型: {model_name}")
                            model, metrics = first_stage_cuml_enhanced(
                                X_gpu, y_gpu,
                                task_type='classification',
                                model_name=model_name
                            )
                            if model is not None and metrics is not None:
                                y_models[model_name] = model
                                y_metrics[model_name] = metrics
                                y_comparator.add_model(model_name, metrics)
                        except Exception as e:
                            print(f"训练Y模型 {model_name} 失败: {str(e)}")
                            traceback.print_exc()
                            y_metrics[model_name] = None
                   
                    # 打印比较结果
                    y_comparator.print_comparison()
                   
                    # 绘制Y模型极坐标图
                    if y_metrics:
                        y_df_data = []
                        for model_name, metrics in y_metrics.items():
                            if metrics is not None:
                                y_df_data.append({
                                    'Accuracy': metrics.get('accuracy', np.nan),
                                    'Precision': metrics.get('precision', np.nan),
                                    'Recall': metrics.get('recall', np.nan),
                                    'F1': metrics.get('f1', np.nan),
                                    'AUC': metrics.get('auc', np.nan),
                                    'Model': model_name
                                })
                        if y_df_data:
                            y_results_df = pd.DataFrame(y_df_data)
                            plot_polar_bars(y_results_df, 5, len(y_df_data), save_path, f"{group_config['prefix']}y_models_polar_{combo_id}")
                   
                    # 训练并比较T模型（分类任务）
                    print("\n=== 训练并比较T模型（分类任务）===")
                    t_models = {}
                    t_metrics = {}
                    for model_name in model_names:
                        try:
                            print(f"\n训练T模型: {model_name}")
                            model, metrics = first_stage_cuml_enhanced(
                                X_gpu, t_gpu,
                                task_type='classification',
                                model_name=model_name
                            )
                            if model is not None and metrics is not None:
                                t_models[model_name] = model
                                t_metrics[model_name] = metrics
                                t_comparator.add_model(model_name, metrics)
                        except Exception as e:
                            print(f"训练T模型 {model_name} 失败: {str(e)}")
                            traceback.print_exc()
                            t_metrics[model_name] = None
                   
                    # 打印比较结果
                    t_comparator.print_comparison()
                   
                    # 绘制T模型极坐标图
                    if t_metrics:
                        t_df_data = []
                        for model_name, metrics in t_metrics.items():
                            if metrics is not None:
                                t_df_data.append({
                                    'Accuracy': metrics.get('accuracy', np.nan),
                                    'Precision': metrics.get('precision', np.nan),
                                    'Recall': metrics.get('recall', np.nan),
                                    'F1': metrics.get('f1', np.nan),
                                    'AUC': metrics.get('auc', np.nan),
                                    'Model': model_name
                                })
                        if t_df_data:
                            t_results_df = pd.DataFrame(t_df_data)
                            plot_polar_bars(t_results_df, 5, len(t_df_data), save_path, f"{group_config['prefix']}t_models_polar_{combo_id}")
                   
                    # 选择最佳模型
                    best_y_model_name = y_comparator.get_best_model()
                    best_t_model_name = t_comparator.get_best_model()
                   
                    if best_y_model_name is None or best_t_model_name is None:
                        print(f"组合 {combo_id}: 未能成功训练任何模型，跳过DML分析")
                        continue
                   
                    print(f"\n组合 {combo_id} 选择的最佳Y模型: {best_y_model_name}")
                    print(f"组合 {combo_id} 选择的最佳T模型: {best_t_model_name}")
                   
                    # 获取最佳模型
                    best_model_y = y_models[best_y_model_name]
                    best_model_t = t_models[best_t_model_name]
                    best_metrics_y = y_comparator.model_metrics[best_y_model_name]
                    best_metrics_t = t_comparator.model_metrics[best_t_model_name]
                   
                    # 如果最佳模型不存在，重新训练
                    if best_model_y is None:
                        print(f"重新训练Y模型: {best_y_model_name}")
                        best_model_y, _ = first_stage_cuml_enhanced(
                            X_gpu, y_gpu,
                            task_type='classification',
                            model_name=best_y_model_name
                        )
                   
                    if best_model_t is None:
                        print(f"重新训练T模型: {best_t_model_name}")
                        best_model_t, _ = first_stage_cuml_enhanced(
                            X_gpu, t_gpu,
                            task_type='classification',
                            model_name=best_t_model_name
                        )
                   
                    # 执行完整的第二阶段DML分析
                    print(f"\n组合 {combo_id} 步骤3: 执行完整的第二阶段DML分析")
                    dml_results = torch_dml_with_pretrained(
                        X, y, t,
                        best_model_y,
                        best_model_t,
                        use_robust_se=True,
                        robust_method='bootstrap',
                        n_bootstrap=1000,
                        n_splits=fold
                    )
                   
                    # 执行敏感性分析（使用优化版）
                    print(f"\n组合 {combo_id} 步骤4: 执行敏感性分析（优化版）")
                    sensitivity_analyzer = SensitivityAnalysisOptimized(
                        X, y, t, control_names
                    )
                   
                    # 定义敏感性分析场景
                    variations = {
                        "基准模型": {
                            "control_vars": control_names,
                            "use_robust_se": True,
                            "robust_method": "bootstrap"
                        },
                        "简化控制变量": {
                            "control_vars": ['contracts', 'ln_total_asset', 'age', 'cfpsedu', 'familysize'],
                            "use_robust_se": True,
                            "robust_method": "bootstrap"
                        },
                        "不同稳健方法": {
                            "control_vars": control_names,
                            "use_robust_se": True,
                            "robust_method": "hc1"
                        }
                    }
                   
                    # 运行敏感性分析
                    sensitivity_results = sensitivity_analyzer.run_analysis(variations, n_splits=fold)
                   
                    # 总结敏感性分析结果
                    try:
                        sensitivity_summary = sensitivity_analyzer.summarize()
                        print(sensitivity_summary)
                    except AttributeError as e:
                        print(f"警告: 无法生成敏感性分析总结，方法可能未实现: {e}")
                        sensitivity_summary = "总结不可用"
                    except Exception as e:
                        print(f"生成敏感性分析总结时出错: {e}")
                        sensitivity_summary = "总结生成出错"
                   
                    # 保存结果到Word
                    print(f"\n组合 {combo_id} 步骤5: 保存结果")
                    save_detailed_results(
                        dml_results['effect'], dml_results['se'], dml_results['lb'], dml_results['ub'],
                        dml_results['t_statistic'], dml_results['p_value'], dml_results['f_statistic'], dml_results['f_p_value'],
                        best_model_y, best_model_t,
                        best_metrics_y, best_metrics_t,
                        sensitivity_results,
                        save_path,
                        robust_method='bootstrap',
                        batch_id=combo_id,
                        prefix=group_config['prefix']
                    )
                   
                    # 收集结果用于综合Word报告
                    group_results[group_counter] = {
                        'y_metrics': y_metrics,
                        't_metrics': t_metrics,
                        'best_y_model': best_y_model_name,
                        'best_t_model': best_t_model_name,
                        'dml_results': dml_results,
                        'sensitivity_results': sensitivity_results,
                        'sensitivity_summary': sensitivity_summary
                    }
                    group_counter += 1
           
            # 生成大组的Word报告
            generate_word_report(group_results, save_path, prefix=group_config['prefix'])
           
            # 收集到所有结果
            all_group_results[group_name] = group_results
       
        print("分析完成！结果已保存并生成Word报告。")
       
    except Exception as e:
        print(f"程序执行出错: {str(e)}")
        traceback.print_exc()
        with open(os.path.join(save_path, "error.log"), "a", encoding='utf-8') as log_file:
            log_file.write(f"[{datetime.now()}] 错误信息: {str(e)}\n")
       
    finally:
        print("执行终极资源清理...")
        clear_gpu_resources()
       
        if torch.cuda.is_available():
            print("最终GPU内存状态:")
            print(f"已用内存: {torch.cuda.memory_allocated()/1e9:.2f} GB")
            print("保留内存: {torch.cuda.memory_reserved()/1e9:.2f} GB")
           
        print("资源清理完成，程序退出！")
